# gpu-batch-serve on a Colab T4

Measures what dynamic batching and FP16 actually buy when serving an
embedding model on a GPU.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

Total runtime is roughly 25-35 minutes for the full grid.

## 1. Confirm a GPU is attached

If this cell prints no GPU, stop and change the runtime type.

In [ ]:
!nvidia-smi
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## 2. Get the code

Replace the URL if the repository is not public yet; otherwise upload the folder to the session and skip this cell.

In [ ]:
import os

REPO_URL = 'https://github.com/aishsunil06/gpu-batch-serve.git'

if not os.path.isdir('gpu-batch-serve'):
    !git clone --depth 1 $REPO_URL
%cd gpu-batch-serve
!ls

## 3. Install dependencies

Colab already ships a CUDA build of torch, so only the serving stack is installed here.

In [ ]:
!pip install -q fastapi uvicorn httpx pydantic transformers pytest pytest-asyncio

## 4. Run the tests first

The benchmark's claims rest on the batcher doing what its configuration says.
If these fail, the numbers below mean nothing, so they run before the sweep.

In [ ]:
!python -m pytest tests/ -q

## 5. Smoke test the harness

Two arms, one repeat, short windows. Confirms the server starts on GPU and
the load generator connects, before committing to the full grid.

In [ ]:
!python scripts/run_bench.py --device cuda --quick --arms A,C --concurrencies 1,8

## 6. Full sweep

Five arms times four concurrency levels times three repeats. Three repeats is
what makes the standard errors in the results table mean anything, so do not
drop to one to save time unless the session is about to be reclaimed.

In [ ]:
!python scripts/run_bench.py --device cuda --repeats 3 --duration 20 --warmup 5

## 7. Read the table

In [ ]:
print(open('bench/results/latest.md').read())

## 8. Download the results

Save these: a Colab session is reclaimed and the raw JSON is the only record
of what produced the table.

In [ ]:
from google.colab import files

files.download('bench/results/latest.json')
files.download('bench/results/latest.md')